In [1]:
# Librerias necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chisquare, pearsonr, ks_2samp
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import os
import json
import pickle

In [2]:
# Nuevos datos con requerimientos de las llegadas
with open('llegadas.pkl', 'rb') as file:
    llegadas_n = pickle.load(file)

In [3]:
# Paso el archivo de llegadas al mismo formato que usaba la funcion datos_llegadas
new_rows = []
requerimiento_dict = {1: "OR", 2: "ICU", 3: "SDU_WARD"}
"""
# nombre_unidades: nombre de las unidades
dict_unidades = {
    "OR": 1,
    "ICU": 2,
    "SDU/WARD": 3,
    "GA": 4,
    "ED": 5
}"""
# Paso el archivo de llegadas al mismo formato que usaba la funcion datos_llegadas
ciclos = len(llegadas_n["WL"])
for llegada in ["WL", "ED"]:
    for ciclo in range(1, ciclos + 1):
        for requerimiento in range(1, 4): # 1: "OR", 2: "ICU", 3: "SDU_WARD"
                    if llegada == "WL":
                        for grd in range(5,9):
                            for repeticiones in range(llegadas_n[llegada][ciclo][(grd, requerimiento)]):
                                new_row = {
                                    'MS_GRD': grd,
                                    'TI': ciclo * 12,
                                    'HOSPITAL': 0, # hospital cero no existe, pero es para poder iterar sin errores
                                    'LLEGADA': llegada,
                                    'REQUERIMIENTO': requerimiento_dict[requerimiento]
                                }
                                new_rows.append(new_row)
                    elif llegada == "ED":
                        for hospital in range(1, 4):
                            for grd in range(1,5):
                                for repeticiones in range(llegadas_n[llegada][ciclo][(hospital, grd, requerimiento)]):
                                    new_row = {
                                        'MS_GRD': grd,
                                        'TI': ciclo * 12,
                                        'HOSPITAL': hospital,
                                        'LLEGADA': llegada,
                                        'REQUERIMIENTO': requerimiento_dict[requerimiento]
                                    }
                                    new_rows.append(new_row)

# Crear un nuevo DataFrame con las nuevas filas
new_llegadas = pd.DataFrame(new_rows)
                    

In [4]:
new_llegadas.head()

,MS_GRD,TI,HOSPITAL,LLEGADA,REQUERIMIENTO
0,6,12,0,WL,OR
1,6,12,0,WL,OR
2,7,12,0,WL,OR
3,5,12,0,WL,ICU
4,5,12,0,WL,ICU


In [5]:
df = new_llegadas.copy()

# Count of entries per hospital
counts = df['HOSPITAL'].value_counts()

# Percentage of entries per hospital
percentages = df['HOSPITAL'].value_counts(normalize=True) * 100

# Combine both into a single DataFrame
summary = pd.DataFrame({'Count': counts, 'Percentage': percentages.round(2)})
print(summary)

          Count  Percentage
HOSPITAL                   
0         78785       50.33
1         30736       19.63
2         26576       16.98
3         20442       13.06
